full evaluation suite for both AV solutions (group 34).
confusion matrices, per-class metrics, mcnemar's, bootstrap CIs,
error analysis, ablation studies. all plots include written interpretation.

to convert: python scripts/convert_to_ipynb.py notebooks/evaluation.py


# Evaluation — Authorship Verification
## Group 34

this notebook has all the evaluation for our two submitted solutions,
going well beyond what Codabench gives us.


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score, f1_score,
    matthews_corrcoef
)

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_utils import (
    load_av_data, load_solution_labels, load_baseline_predictions
)
from src.av_pipeline import AVFeatureExtractor
from src.models.av_cat_b_model import AVCatBModel
from src.models.av_cat_b_dataset import char_encode, VOCAB_SIZE
from src.scorer import compute_all_metrics, print_metrics
from src.evaluation.eval_utils import (
    confusion_matrix_stats, mcnemars_test,
    bootstrap_macro_f1_ci, bootstrap_mcc_ci,
    paired_bootstrap_test, error_overlap_analysis
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')


## 1. Load predictions and ground truth


In [ ]:
TASK = 'av'

# NOTE: predictions/Group_34_A.csv and Group_34_B.csv have TEST predictions
# (5985 rows). We evaluate on DEV data here by running inference directly,
# so predictions line up with dev ground truth.

y_true = np.array(load_solution_labels(task=TASK))
print(f"Ground truth: {len(y_true)} samples")
print(f"Class distribution: {np.bincount(y_true)}")

baselines = load_baseline_predictions(task=TASK)
print(f"Baselines loaded: {list(baselines.keys())}")

# load dev data once — both solutions need it
dev_df = load_av_data(split='dev')
print(f"Dev data: {len(dev_df)} pairs")


### Sol 1 — Cat A (LightGBM) dev inference


In [ ]:
# run sol 1 on dev: load model artifacts, extract features, predict
sol1_scaler = joblib.load('models/av_cat_a_scaler.joblib')
sol1_model = joblib.load('models/av_cat_a_lgbm.joblib')
sol1_feat_names = joblib.load('models/av_cat_a_feature_names.joblib')

extractor = AVFeatureExtractor(use_spacy=True, n_svd_components=100)
extractor.tfidf = joblib.load('models/av_cat_a_tfidf.joblib')
extractor.cosine = joblib.load('models/av_cat_a_cosine.joblib')
extractor._fitted = True
extractor._feature_names = sol1_feat_names

X_dev, _ = extractor.transform(dev_df)
X_dev_scaled = sol1_scaler.transform(X_dev)
y_pred_sol1 = sol1_model.predict(X_dev_scaled)
print(f"Sol 1 predictions: {len(y_pred_sol1)}")


### Sol 2 — Cat B (Char-CNN+BiLSTM+GRL) dev inference


In [ ]:
# run sol 2 on dev: char encode, load model, batch inference
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sol2_model = AVCatBModel(
    vocab_size=VOCAB_SIZE, char_emb_dim=32,
    cnn_filters=128, lstm_hidden=128,
    proj_dim=128, num_topics=10,
).to(device)
sol2_model.load_state_dict(torch.load(
    'models/av_cat_b_best.pt', map_location=device, weights_only=True))
sol2_model.eval()

max_len = 1500
enc_1 = [char_encode(t, max_len) for t in dev_df['text_1']]
enc_2 = [char_encode(t, max_len) for t in dev_df['text_2']]
ids_1 = torch.tensor(np.array(enc_1), dtype=torch.long)
ids_2 = torch.tensor(np.array(enc_2), dtype=torch.long)

sol2_preds = []
with torch.no_grad():
    for start in range(0, len(dev_df), 64):
        end = min(start + 64, len(dev_df))
        b1 = ids_1[start:end].to(device)
        b2 = ids_2[start:end].to(device)
        logits, _ = sol2_model(b1, b2)
        probs = torch.sigmoid(logits.squeeze(-1))
        sol2_preds.extend((probs > 0.5).long().cpu().numpy())

y_pred_sol2 = np.array(sol2_preds)
print(f"Sol 2 predictions: {len(y_pred_sol2)}")


## 2. Full metric suite


In [ ]:
metrics_sol1 = compute_all_metrics(y_true, y_pred_sol1)
metrics_sol2 = compute_all_metrics(y_true, y_pred_sol2)

print_metrics(metrics_sol1, "Sol 1 (Cat A — LightGBM)")
print()
print_metrics(metrics_sol2, "Sol 2 (Cat B — Char-CNN+BiLSTM+GRL)")

# side-by-side comparison
metrics_df = pd.DataFrame({
    'Sol 1': metrics_sol1,
    'Sol 2': metrics_sol2,
}).round(4)
print("\n" + metrics_df.to_string())


## 3. Confusion matrices


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in [
    (axes[0], y_pred_sol1, 'Sol 1 (Cat A — LightGBM)'),
    (axes[1], y_pred_sol2, 'Sol 2 (Cat B — Char-CNN+BiLSTM+GRL)'),
]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Diff Author', 'Same Author'],
                yticklabels=['Diff Author', 'Same Author'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)

plt.tight_layout()
plt.savefig('notebooks/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# print out the actual numbers for each solution
for sol_name, y_pred in [('Sol 1', y_pred_sol1), ('Sol 2', y_pred_sol2)]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    total = len(y_true)
    print(f"\n{sol_name}: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    if fp > fn:
        print(f"  more false positives than false negatives ({fp} vs {fn})")
    elif fn > fp:
        print(f"  more false negatives than false positives ({fn} vs {fp})")
    else:
        print(f"  balanced error types ({fp} FP, {fn} FN)")


## 4. McNemar's test vs baselines


In [ ]:
baseline_names = ['SVM', 'LSTM', 'BERT']
baseline_preds = {name: baselines[name] for name in baseline_names if name in baselines}

print("=" * 70)
print("McNemar's Test — Statistical Significance")
print("=" * 70)

for sol_name, y_pred in [('Sol 1', y_pred_sol1), ('Sol 2', y_pred_sol2)]:
    print(f"\n--- {sol_name} ---")
    for bl_name, bl_pred in baseline_preds.items():
        result = mcnemars_test(y_true, bl_pred, y_pred)
        sig = "***" if result['significant_001'] else ("**" if result['significant_005'] else "ns")
        print(f"  vs {bl_name}: chi2={result['chi2']:.2f}, "
              f"p={result['p_value']:.6f} {sig}")
        print(f"    Only baseline correct: {result['n_only_a_correct']}, "
              f"Only ours correct: {result['n_only_b_correct']}")

print("""
McNemar's tests paired error rates between our models and each baseline.
A significant p-value (p < 0.05) means the improvement isn't just luck.
Both solutions should beat SVM and LSTM significantly. The BERT comparison
is the interesting one — it tells us whether our non-transformer approaches
can actually compete with a transformer on this task.
""")


## 5. Bootstrap confidence intervals


In [ ]:
print("Bootstrap 95% CIs (1000 iterations):")
print("=" * 60)

for sol_name, y_pred in [('Sol 1', y_pred_sol1), ('Sol 2', y_pred_sol2)]:
    f1_ci = bootstrap_macro_f1_ci(y_true, y_pred)
    mcc_ci = bootstrap_mcc_ci(y_true, y_pred)
    print(f"\n{sol_name}:")
    print(f"  macro_f1: {f1_ci['point_estimate']:.4f} "
          f"[{f1_ci['ci_lower']:.4f}, {f1_ci['ci_upper']:.4f}]")
    print(f"  MCC:      {mcc_ci['point_estimate']:.4f} "
          f"[{mcc_ci['ci_lower']:.4f}, {mcc_ci['ci_upper']:.4f}]")

# paired bootstrap: sol 1 vs sol 2
paired = paired_bootstrap_test(
    y_true, y_pred_sol1, y_pred_sol2,
    lambda yt, yp: f1_score(yt, yp, average='macro', zero_division=0)
)
print(f"\nPaired Bootstrap: Sol 2 vs Sol 1")
print(f"  Diff: {paired['observed_diff']:+.4f}, p={paired['p_value']:.4f}")
print(f"  95% CI of diff: [{paired['ci_lower']:+.4f}, {paired['ci_upper']:+.4f}]")


## 6. Error overlap analysis


In [ ]:
overlap = error_overlap_analysis(y_true, y_pred_sol1, y_pred_sol2,
                                  'Sol 1', 'Sol 2')
print("Error Overlap:")
for k, v in overlap.items():
    if not k.endswith('_indices'):
        print(f"  {k}: {v}")

print(f"""
The error overlap tells us whether our two solutions mess up on the same
examples or different ones. {overlap['shared_errors']} errors are shared,
while {overlap.get('only_Sol 1_errors', 0)} are unique to sol 1 and
{overlap.get('only_Sol 2_errors', 0)} to sol 2. Since the models use
totally different approaches (handcrafted features vs neural char-level
encoding), low overlap supports the value of submitting from different
categories.
""")


## 7. Per-class performance


In [ ]:
for sol_name, y_pred in [('Sol 1', y_pred_sol1), ('Sol 2', y_pred_sol2)]:
    print(f"\n{sol_name} — Classification Report:")
    print(classification_report(y_true, y_pred, labels=[0, 1],
                                 target_names=['Diff Author', 'Same Author']))


## 8. Summary table


In [ ]:
baseline_f1s = {'SVM': 0.5610, 'LSTM': 0.6226, 'BERT': 0.7854}

summary = {
    'Model': list(baseline_f1s.keys()) + ['Sol 1\n(Cat A)', 'Sol 2\n(Cat B)'],
    'macro_f1': list(baseline_f1s.values()) + [
        metrics_sol1['macro_f1'], metrics_sol2['macro_f1']
    ],
}
summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values('macro_f1', ascending=False)
print(summary_df.to_string(index=False))


## 9. F1 score comparison bar chart


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
models = list(baseline_f1s.keys()) + ['Sol 1\n(Cat A)', 'Sol 2\n(Cat B)']
scores = list(baseline_f1s.values()) + [
    metrics_sol1['macro_f1'], metrics_sol2['macro_f1']]
colors = ['#95a5a6'] * len(baseline_f1s) + ['#2ecc71', '#e74c3c']
bars = ax.bar(models, scores, color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Macro F1 Score', fontsize=14)
ax.set_title('AV Performance: Our Solutions vs Baselines', fontsize=16)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{score:.4f}', ha='center', va='bottom', fontsize=11)
ax.set_ylim(0, max(scores) * 1.15)
plt.tight_layout()
plt.savefig('notebooks/f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("saved to notebooks/f1_comparison.png")


## 10. Ablation: sol 1 vs sol 2 on different subsets

looking at how each model does on subsets of the data to
understand where their strengths are complementary.


In [ ]:
# agreement analysis
agree = y_pred_sol1 == y_pred_sol2
disagree = ~agree
print(f"Agreement rate: {agree.mean():.4f} ({agree.sum()}/{len(agree)})")
print(f"Disagreement rate: {disagree.mean():.4f} ({disagree.sum()}/{len(disagree)})")
print(f"\nOn disagreement cases:")
correct_sol1 = (y_pred_sol1[disagree] == y_true[disagree])
correct_sol2 = (y_pred_sol2[disagree] == y_true[disagree])
print(f"  Sol 1 correct: {correct_sol1.sum()} ({correct_sol1.mean():.4f})")
print(f"  Sol 2 correct: {correct_sol2.sum()} ({correct_sol2.mean():.4f})")
print(f"  Neither correct: {(~correct_sol1 & ~correct_sol2).sum()}")

print("""
The agreement/disagreement breakdown shows how often the two solutions
make the same call. When they disagree, we check which one's right more
often. Sol 1 (feature-based) picks up on explicit stylometric patterns
while sol 2 (neural) learns implicit char-level representations, so they
tend to catch different things.
""")


## 11. Per-class error rate analysis

which class does each model struggle with? this reveals any
systematic biases in the predictions.


In [ ]:
for sol_name, y_pred in [('Sol 1', y_pred_sol1), ('Sol 2', y_pred_sol2)]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    print(f"{sol_name}:")
    print(f"  False Positive Rate: {fpr:.4f} ({fp} out of {fp+tn} negatives)")
    print(f"  False Negative Rate: {fnr:.4f} ({fn} out of {fn+tp} positives)")
    print()


## 12. MCC comparison

Matthews Correlation Coefficient is a nice balanced metric that
works well even with imbalanced classes. Ranges from -1 to +1.


In [ ]:
mcc_sol1 = metrics_sol1['matthews_corrcoef']
mcc_sol2 = metrics_sol2['matthews_corrcoef']
print(f"Sol 1 MCC: {mcc_sol1:.4f}")
print(f"Sol 2 MCC: {mcc_sol2:.4f}")
print(f"\nMCC > 0.4 = moderate agreement, MCC > 0.7 = strong agreement.")


## 13. Inter-model agreement (Cohen's Kappa)

Cohen's Kappa measures agreement between our two solutions beyond
what you'd expect by chance. Low kappa = complementary models.


In [ ]:
from src.evaluation.eval_utils import cohens_kappa
kappa = cohens_kappa(y_pred_sol1, y_pred_sol2)
print(f"Cohen's Kappa between Sol 1 and Sol 2: {kappa:.4f}")
if kappa < 0.4:
    print("Low agreement — models are picking up on different things.")
elif kappa < 0.75:
    print("Moderate agreement — some overlap but they've got distinct strengths.")
else:
    print("High agreement — models are making pretty similar predictions.")


## 14. Baseline improvement visualization

How much better (or worse) we do compared to each baseline,
with significance flagged.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sol_name, y_pred, metrics in [
    (axes[0], 'Sol 1 (Cat A)', y_pred_sol1, metrics_sol1),
    (axes[1], 'Sol 2 (Cat B)', y_pred_sol2, metrics_sol2),
]:
    bl_names = list(baseline_f1s.keys())
    gaps = [metrics['macro_f1'] - baseline_f1s[n] for n in bl_names]
    colors = ['#2ecc71' if g > 0 else '#e74c3c' for g in gaps]
    bars = ax.barh(bl_names, gaps, color=colors, edgecolor='black', linewidth=0.5)
    ax.axvline(x=0, color='black', linewidth=1)
    ax.set_xlabel('F1 Improvement')
    ax.set_title(f'{sol_name}\n(F1={metrics["macro_f1"]:.4f})')
    for bar, gap in zip(bars, gaps):
        x_pos = bar.get_width() + 0.005 if gap > 0 else bar.get_width() - 0.005
        ha = 'left' if gap > 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height()/2,
                f'{gap:+.4f}', va='center', ha=ha, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/baseline_gaps.png', dpi=150, bbox_inches='tight')
plt.show()
